In [1]:
!pip install -U fastapi uvicorn nest_asyncio sqlalchemy psycopg2-binary pandas numpy transformers accelerate sentencepiece sentence-transformers tqdm
!pip install faiss-cpu --no-deps

   ---------------------------------------- 0.0/588.7 kB ? eta -:--:--
   ---------------------------------------- 588.7/588.7 kB 9.1 MB/s  0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.4.1
    Uninstalling sentence-transformers-5.4.1:
      Successfully uninstalled sentence-transformers-5.4.1


In [4]:
# =========================================================
# DBR LLM + FAISS 추천 API
# - Jupyter Notebook 실행용 버전
# - 사용자 쿼리 + LLM 확장 문장 합쳐서 임베딩
# - FAISS로 빠르게 후보 검색
# - LLM JSON 실패 시 규칙 기반 query_meta 자동 생성
# - LLM 결과가 DB 값과 다르면 정규화
# - final_score는 sim_score 중심으로 계산
# =========================================================

import json
from typing import Optional

import faiss
import nest_asyncio
import numpy as np
import pandas as pd
import torch
import uvicorn

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer
from sqlalchemy import create_engine
from transformers import AutoTokenizer, AutoModelForCausalLM


nest_asyncio.apply()


# =========================================================
# DB 연결 정보
# =========================================================
DB_CONFIG = {
    "host": "project-db-campus.smhrd.com",
    "port": 3310,
    "dbname": "cgi_25k_da2_p3_3",
    "user": "cgi_25k_da2_p3_3",
    "password": "smhrd3"
}

DATABASE_URL = (
    f"postgresql+psycopg2://"
    f"{DB_CONFIG['user']}:{DB_CONFIG['password']}@"
    f"{DB_CONFIG['host']}:{DB_CONFIG['port']}/"
    f"{DB_CONFIG['dbname']}"
)

engine = create_engine(DATABASE_URL, pool_pre_ping=True)


# =========================================================
# FastAPI 앱 생성
# =========================================================
app = FastAPI(title="DBR LLM + FAISS Recommendation API")


# =========================================================
# CORS 설정
# =========================================================
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


# =========================================================
# 전역 변수
# =========================================================
df: Optional[pd.DataFrame] = None
index: Optional[faiss.IndexFlatIP] = None
model: Optional[SentenceTransformer] = None
tokenizer = None
llm_model = None


# =========================================================
# 요청 데이터 형식
# =========================================================
class RecommendRequest(BaseModel):
    query_text: str


# =========================================================
# DB 데이터 로드
# =========================================================
def load_case_data() -> pd.DataFrame:
    query = """
    SELECT
        case_idx,
        chapter_title,
        title,
        summary,
        src_url,
        issue_no,
        pub_year,
        comp_name,
        comp_size,
        industry,
        prob_main,
        prob_keyword,
        prob_def,
        sol_type,
        sol_detail,
        perf_type,
        perf_dir,
        embedding,
        x,
        y
    FROM t_case
    ORDER BY case_idx
    """
    return pd.read_sql(query, engine)


# =========================================================
# embedding 변환
# =========================================================
def parse_embedding(value):
    if value is None:
        return None

    if isinstance(value, np.ndarray):
        return value.astype("float32")

    if isinstance(value, (list, tuple)):
        return np.array(value, dtype="float32")

    if isinstance(value, str):
        value = value.strip()

        if value.startswith("[") and value.endswith("]"):
            value = value[1:-1]

        return np.array([float(x) for x in value.split(",")], dtype="float32")

    raise TypeError(f"지원하지 않는 embedding 타입: {type(value)}")


# =========================================================
# JSON 응답 안전 처리
# =========================================================
def clean_records(records):
    cleaned = []

    for row in records:
        new_row = {}

        for key, value in row.items():
            if pd.isna(value):
                new_row[key] = None
            elif isinstance(value, np.integer):
                new_row[key] = int(value)
            elif isinstance(value, np.floating):
                new_row[key] = float(value)
            else:
                new_row[key] = value

        cleaned.append(new_row)

    return cleaned


# =========================================================
# 기본 query_meta
# =========================================================
def get_default_query_meta(query_text=""):
    return {
        "prob_main": "",
        "prob_keyword": [],
        "expected_cause": "",
        "perf_type": "",
        "sol_type": "",
        "industry": None,
        "expanded_query": query_text
    }


# =========================================================
# LLM 실패 시 사용할 규칙 기반 query_meta
# =========================================================
def rule_based_query_meta(query_text):
    meta = get_default_query_meta(query_text)

    if any(word in query_text for word in ["AI", "인공지능", "자동화", "디지털", "데이터"]):
        meta["prob_main"] = "혁신"
        meta["prob_keyword"] = ["디지털전환", "기술전환필요", "생산성"]
        meta["sol_type"] = "기술 도입"

    elif any(word in query_text for word in ["브랜드", "인지도", "마케팅", "광고", "신규 고객", "고객 유입"]):
        meta["prob_main"] = "고객"
        meta["prob_keyword"] = ["브랜딩", "마케팅", "참여도"]
        meta["sol_type"] = "마케팅·브랜딩"

    elif any(word in query_text for word in ["재방문", "이탈", "리텐션", "구독", "유지"]):
        meta["prob_main"] = "고객"
        meta["prob_keyword"] = ["이탈률", "참여도", "만족도"]
        meta["sol_type"] = "사용자 유지"

    elif any(word in query_text for word in ["생산성", "효율", "비용", "업무", "운영", "복잡"]):
        meta["prob_main"] = "효율"
        meta["prob_keyword"] = ["생산성", "운영복잡도", "비용절감"]
        meta["sol_type"] = "운영 효율화"

    elif any(word in query_text for word in ["성장", "확장", "해외", "신규시장", "시장 진출"]):
        meta["prob_main"] = "성장"
        meta["prob_keyword"] = ["신규시장", "확장성"]
        meta["sol_type"] = "제품·서비스 개선"

    return meta


# =========================================================
# LLM 결과 정규화
# - LLM이 DB에 없는 값을 만들면 DB 기준 값으로 변환
# =========================================================
def normalize_query_meta(result, query_text=""):
    valid_prob_main = ["고객", "성장", "효율", "혁신"]

    valid_sol_type = [
        "운영 효율화",
        "제품·서비스 개선",
        "마케팅·브랜딩",
        "사용자 유지",
        "수익화",
        "플랫폼 활용",
        "기술 도입"
    ]

    valid_keywords = [
        "이탈률", "참여도", "만족도", "접근성", "신규시장", "확장성",
        "디지털전환", "기술전환필요", "생산성", "개인화", "운영복잡도",
        "비용절감", "물류", "팬덤", "브랜딩", "마케팅"
    ]

    prob_main_mapping = {
        "신규시장": "성장",
        "신규 시장": "성장",
        "브랜드": "고객",
        "브랜딩": "고객",
        "마케팅": "고객",
        "수익화": "성장",
        "고객 유입": "고객"
    }

    keyword_mapping = {
        "브랜드 인지도": "브랜딩",
        "브랜드인지도": "브랜딩",
        "신규 고객": "참여도",
        "고객 유입": "참여도",
        "신규고객": "참여도",
        "AI": "디지털전환",
        "인공지능": "디지털전환",
        "업무 생산성": "생산성",
        "프로세스": "운영복잡도",
        "운영 효율": "생산성"
    }

    sol_type_mapping = {
        "messaging and marketing": "마케팅·브랜딩",
        "marketing": "마케팅·브랜딩",
        "프로모션 및 마케팅": "마케팅·브랜딩",
        "브랜드 마케팅": "마케팅·브랜딩",
        "technology adoption": "기술 도입",
        "AI 도입": "기술 도입",
        "platform": "플랫폼 활용",
        "operating efficiency": "운영 효율화"
    }

    # 빈 값이면 규칙 기반으로 생성
    if not result.get("prob_main") and not result.get("sol_type") and not result.get("prob_keyword"):
        return rule_based_query_meta(query_text)

    # prob_main 정규화
    prob_main = result.get("prob_main", "")
    if prob_main in prob_main_mapping:
        result["prob_main"] = prob_main_mapping[prob_main]

    if result.get("prob_main") not in valid_prob_main:
        result["prob_main"] = rule_based_query_meta(query_text).get("prob_main", "")

    # sol_type 정규화
    sol_type = result.get("sol_type", "")
    if sol_type in sol_type_mapping:
        result["sol_type"] = sol_type_mapping[sol_type]

    if result.get("sol_type") not in valid_sol_type:
        result["sol_type"] = rule_based_query_meta(query_text).get("sol_type", "")

    # prob_keyword 정규화
    keywords = result.get("prob_keyword", [])

    if not isinstance(keywords, list):
        keywords = [keywords]

    normalized_keywords = []

    for keyword in keywords:
        if keyword in keyword_mapping:
            normalized_keywords.append(keyword_mapping[keyword])
        elif keyword in valid_keywords:
            normalized_keywords.append(keyword)

    # 키워드가 하나도 없으면 규칙 기반 키워드 사용
    if not normalized_keywords:
        normalized_keywords = rule_based_query_meta(query_text).get("prob_keyword", [])

    result["prob_keyword"] = normalized_keywords

    result.setdefault("expected_cause", "")
    result.setdefault("perf_type", "")
    result.setdefault("industry", None)
    result.setdefault("expanded_query", query_text)

    return result


# =========================================================
# LLM 응답에서 JSON 추출
# - JSON 파싱 실패 시 규칙 기반 query_meta로 대체
# =========================================================
def extract_last_json_from_text(text, query_text=""):
    try:
        start = text.find("{")
        end = text.rfind("}")

        if start == -1 or end == -1:
            raise ValueError("JSON 시작/끝을 찾지 못했습니다.")

        json_text = text[start:end + 1]
        result = json.loads(json_text)

        if "prob_keyword" not in result:
            result["prob_keyword"] = []

        if not isinstance(result.get("prob_keyword"), list):
            result["prob_keyword"] = [result.get("prob_keyword")]

        result.setdefault("prob_main", "")
        result.setdefault("expected_cause", "")
        result.setdefault("perf_type", "")
        result.setdefault("sol_type", "")
        result.setdefault("industry", None)
        result.setdefault("expanded_query", query_text)

        result = normalize_query_meta(result, query_text)

        return result

    except Exception:
        print("LLM 원본 응답:", text)
        print("JSON 파싱 실패 → 규칙 기반 query_meta로 추천 진행")
        return rule_based_query_meta(query_text)


# =========================================================
# Qwen LLM으로 사용자 질의 분석
# =========================================================
def analyze_query_with_local_llm(query_text):
    global tokenizer, llm_model

    if tokenizer is None or llm_model is None:
        raise RuntimeError("Qwen LLM 모델이 아직 로드되지 않았습니다.")

    prompt = f"""
너는 DBR 케이스 추천 서비스의 사용자 질의 분석기다.
사용자 입력을 분석해서 JSON 하나만 출력하라.

선택지:
prob_main = 고객, 성장, 효율, 혁신
prob_keyword = 이탈률, 참여도, 만족도, 접근성, 신규시장, 확장성, 디지털전환, 기술전환필요, 생산성, 개인화, 운영복잡도, 비용절감, 물류, 팬덤, 브랜딩, 마케팅
sol_type = 운영 효율화, 제품·서비스 개선, 마케팅·브랜딩, 사용자 유지, 수익화, 플랫폼 활용, 기술 도입
industry = IT·플랫폼, 커머스, 제조, 헬스케어, 콘텐츠·미디어, 금융, 식음료, 리테일, 물류·운송, 부동산·공간, 기타, null

분류 규칙:
- AI, 인공지능, 자동화, 디지털, 데이터 → prob_main은 혁신
- 비용절감, 업무효율, 생산성 → prob_main은 효율
- 고객경험, 재방문, 이탈, 만족도, 참여, 팬덤, 커뮤니티 → prob_main은 고객
- 신규시장, 해외진출, 확장, 성장 → prob_main은 성장
- 마케팅, 광고, 캠페인, 브랜딩 → sol_type은 마케팅·브랜딩
- 플랫폼, 정보 접근성, 검색, 탐색, 정보 제공 → sol_type은 플랫폼 활용
- AI와 기술 도입이 핵심이면 sol_type은 기술 도입
- 재방문, 리텐션, 구독 유지 → sol_type은 사용자 유지

반드시 아래 JSON 형식으로만 출력하라.
JSON 외의 설명, 문장, 코드블록은 절대 출력하지 마라.

{{
  "prob_main": "",
  "prob_keyword": [],
  "expected_cause": "",
  "perf_type": "",
  "sol_type": "",
  "industry": null,
  "expanded_query": ""
}}

expanded_query는 사용자 입력을 DBR 케이스 검색에 적합한 한 문장으로 확장하라.
문제 상황, 해결 방향, 기대 성과를 포함하라.

사용자 입력:
{query_text}
"""

    messages = [{"role": "user", "content": prompt}]

    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        input_text,
        return_tensors="pt"
    ).to(llm_model.device)

    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][input_length:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return extract_last_json_from_text(response, query_text)


# =========================================================
# 최종 점수 계산
# - sim_score 그대로 사용
# - query_meta 기반 보정점수 추가
# =========================================================
def calculate_final_score(row, query_meta):
    score = float(row["sim_score"])

    query_prob_main = query_meta.get("prob_main")
    query_sol_type = query_meta.get("sol_type")
    query_keywords = query_meta.get("prob_keyword", [])
    row_keyword = row["prob_keyword"]

    if not isinstance(query_keywords, list):
        query_keywords = [query_keywords]

    if query_prob_main and row["prob_main"] == query_prob_main:
        score += 0.05

    if row_keyword in query_keywords:
        score += 0.08

    related_keyword_map = {
        "마케팅": ["마케팅", "브랜딩", "참여도", "팬덤", "개인화", "만족도", "확장성"],
        "브랜딩": ["마케팅", "브랜딩", "팬덤", "참여도"],
        "디지털전환": ["디지털전환", "기술전환필요", "개인화", "생산성"],
        "접근성": ["접근성", "UX", "만족도"],
        "이탈률": ["이탈률", "참여도", "만족도"],
        "참여도": ["참여도", "만족도", "팬덤"],
        "생산성": ["생산성", "운영복잡도", "비용절감"],
        "비용절감": ["비용절감", "운영복잡도", "생산성"],
    }

    related_keywords = []

    for keyword in query_keywords:
        related_keywords.extend(related_keyword_map.get(keyword, []))

    if row_keyword in related_keywords:
        score += 0.05

    if query_sol_type and row["sol_type"] == query_sol_type:
        score += 0.07

    if query_meta.get("industry") and row["industry"] == query_meta["industry"]:
        score += 0.03

    return round(min(float(score), 1.0), 5)


# =========================================================
# 추천 함수
# =========================================================
def recommend_cases_faiss_with_llm(query_text, top_k=60, final_k=5):
    global df, index, model

    if df is None or index is None or model is None:
        raise RuntimeError("추천 엔진이 아직 초기화되지 않았습니다.")

    query_meta = analyze_query_with_local_llm(query_text)

    expanded_query = query_meta.get("expanded_query", "")

    if expanded_query:
        search_text = query_text + " " + expanded_query[:120]
    else:
        search_text = query_text

    query_emb = model.encode(
        [search_text],
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    ).astype("float32")

    search_k = min(top_k, len(df))
    scores, indices = index.search(query_emb, search_k)

    candidate_df = df.iloc[indices[0]].copy()
    candidate_df["sim_score"] = scores[0].astype(float)

    candidate_df["final_score"] = candidate_df.apply(
        lambda row: calculate_final_score(row, query_meta),
        axis=1
    )

    result = (
        candidate_df[
            [
                "case_idx",
                "title",
                "summary",
                "src_url",
                "prob_main",
                "prob_keyword",
                "sol_type",
                "industry",
                "x",
                "y",
                "sim_score",
                "final_score"
            ]
        ]
        .sort_values("final_score", ascending=False)
        .head(final_k)
        .reset_index(drop=True)
    )

    result.insert(0, "ranking", range(1, len(result) + 1))
    result = result.replace({np.nan: None})

    return result, query_meta


# =========================================================
# 추천 엔진 로딩
# =========================================================
def load_recommendation_engine():
    global df, index, model, tokenizer, llm_model

    print("DBR LLM + FAISS 추천 엔진 로딩 중...")

    df = load_case_data()

    if len(df) == 0:
        raise RuntimeError("t_case 테이블에 데이터가 없습니다.")

    print("DB 데이터 로드 완료:", len(df), "건")

    df["embedding_array"] = df["embedding"].apply(parse_embedding)

    embeddings = np.vstack(df["embedding_array"].values).astype("float32")

    faiss.normalize_L2(embeddings)

    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)

    print("FAISS 인덱스 생성 완료:", index.ntotal, "개")

    model = SentenceTransformer("jhgan/ko-sbert-sts")
    print("Ko-SBERT 모델 로드 완료")

    qwen_model_name = "Qwen/Qwen2.5-0.5B-Instruct"

    if torch.cuda.is_available():
        torch_dtype = torch.float16
        print("CUDA 사용 가능:", torch.cuda.get_device_name(0))
    else:
        torch_dtype = torch.float32
        print("GPU가 없습니다. CPU 모드로 실행합니다.")

    tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)

    llm_model = AutoModelForCausalLM.from_pretrained(
        qwen_model_name,
        torch_dtype=torch_dtype,
        device_map="auto"
    )

    llm_model.eval()

    print("Qwen LLM 모델 로드 완료:", qwen_model_name)
    print("DBR LLM + FAISS 추천 엔진 로딩 완료")


# =========================================================
# API
# =========================================================
@app.get("/")
def root():
    return {"message": "DBR LLM + FAISS Recommendation API is running"}


@app.get("/cases")
def get_cases():
    try:
        query = """
        SELECT
            case_idx,
            title,
            summary,
            src_url,
            prob_main,
            prob_keyword,
            sol_type,
            industry,
            x,
            y
        FROM t_case
        ORDER BY case_idx
        """

        cases_df = pd.read_sql(query, engine)
        cases_df = cases_df.replace({np.nan: None})

        return {
            "cases": clean_records(cases_df.to_dict(orient="records"))
        }

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/recommend")
def recommend(request: RecommendRequest):
    try:
        if not request.query_text.strip():
            raise HTTPException(status_code=400, detail="query_text를 입력하세요.")

        result, query_meta = recommend_cases_faiss_with_llm(
            query_text=request.query_text,
            top_k=60,
            final_k=5
        )

        return {
            "query_meta": query_meta,
            "recommendations": clean_records(result.to_dict(orient="records"))
        }

    except HTTPException:
        raise

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


# =========================================================
# Jupyter Notebook 실행 코드
# =========================================================
load_recommendation_engine()

config = uvicorn.Config(
    app,
    host="127.0.0.1",
    port=8003,
    log_level="info"
)

server = uvicorn.Server(config)

await server.serve()

DBR LLM + FAISS 추천 엔진 로딩 중...
DB 데이터 로드 완료: 115 건
FAISS 인덱스 생성 완료: 115 개


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Ko-SBERT 모델 로드 완료
GPU가 없습니다. CPU 모드로 실행합니다.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen LLM 모델 로드 완료: Qwen/Qwen2.5-0.5B-Instruct
DBR LLM + FAISS 추천 엔진 로딩 완료


INFO:     Started server process [9580]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8003 (Press CTRL+C to quit)


INFO:     127.0.0.1:56753 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:56753 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:56754 - "POST /recommend HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [9580]


In [9]:
# 깃허브 back-end 브랜치 업로드

!git clone https://github.com/HanGS01/Route-Finders.git

%cd Route-Finders

!git checkout back-end

C:\Users\smhrd\실전 프로젝트 eeee\Route-Finders\Route-Finders\Route-Finders


Cloning into 'Route-Finders'...


branch 'back-end' set up to track 'origin/back-end'.


Switched to a new branch 'back-end'
